# exp_8a — Experiment 8A: LSTM and Bidirectional LSTM for Structured Time-Series Forecasting

Source-based notebook converted from the uploaded experiment code document.


**Note:** The source uses powerconsumption.csv from a Windows dataset folder. Update the path if needed.


In [ ]:
import zipfile

zip_path = "D:\DL DATASET\powerconsumption.csv.zip"   # Change to your file name

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("D:\DL DATASET")

In [ ]:
import os

for root, dirs, files in os.walk("D:\DL DATASET"):
    for file in files:
        print(file)

In [ ]:
import pandas as pd

df = pd.read_csv("D:\DL DATASET\powerconsumption.csv")

print(df.head())
print(df.info())
print(df.columns)
print(df.isnull().sum())

In [ ]:
df["Datetime"] = pd.to_datetime(df["Datetime"])

df = df.sort_values("Datetime")

df.head()

In [ ]:
target = df[["PowerConsumption_Zone1"]]

target.head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

scaled_data = scaler.fit_transform(target)

In [ ]:
train_size = int(len(scaled_data) * 0.8)

train = scaled_data[:train_size]
test = scaled_data[train_size:]

In [ ]:
import numpy as np

def create_sequences(data, lookback):
    X = []
    y = []

    for i in range(len(data) - lookback):
        X.append(data[i:i+lookback])
        y.append(data[i+lookback])

    return np.array(X), np.array(y)

lookback = 30

X_train, y_train = create_sequences(train, lookback)
X_test, y_test = create_sequences(test, lookback)

print(X_train.shape)
print(y_train.shape)

print(X_test.shape)
print(y_test.shape)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

lstm_model = Sequential([

    LSTM(
        64,
        input_shape=(30,1)
    ),

    Dense(32, activation="relu"),

    Dense(1)

])

lstm_model.summary()

In [ ]:
lstm_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [

    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        verbose=1
    )

]

history_lstm = lstm_model.fit(

    X_train,
    y_train,

    validation_data=(X_test,y_test),

    epochs=10,

    batch_size=32,

    callbacks=callbacks

)

In [ ]:
loss_lstm, mae_lstm = lstm_model.evaluate(X_test, y_test, verbose=0)

print("LSTM Loss :", loss_lstm)
print("LSTM MAE  :", mae_lstm)

In [ ]:
from tensorflow.keras.layers import Bidirectional, LSTM, Dense
from tensorflow.keras.models import Sequential

In [ ]:
bilstm_model = Sequential([

    Bidirectional(
        LSTM(64),
        input_shape=(30,1)
    ),

    Dense(32, activation="relu"),

    Dense(1)

])

bilstm_model.summary()

In [ ]:
bilstm_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [ ]:
history_bilstm = bilstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=32,
    callbacks=callbacks
)

In [ ]:
print(callbacks)

In [ ]:
loss_bilstm, mae_bilstm = bilstm_model.evaluate(X_test, y_test, verbose=0)

print("Bidirectional LSTM Loss:", loss_bilstm)
print("Bidirectional LSTM MAE :", mae_bilstm)

In [ ]:
comparison = pd.DataFrame({

    "Model":[
        "LSTM",
        "Bidirectional LSTM"
    ],

    "Loss":[
        loss_lstm,
        loss_bilstm
    ],

    "MAE":[
        mae_lstm,
        mae_bilstm
    ]

})

comparison

In [ ]:
pred_lstm = lstm_model.predict(X_test)

pred_lstm = scaler.inverse_transform(pred_lstm)

actual = scaler.inverse_transform(y_test)

plt.figure(figsize=(15,5))

plt.plot(actual,label="Actual")

plt.plot(pred_lstm,label="Predicted")

plt.legend()

plt.title("LSTM Prediction")

plt.show()

In [ ]:
pred_bilstm = bilstm_model.predict(X_test)

pred_bilstm = scaler.inverse_transform(pred_bilstm)

plt.figure(figsize=(15,5))

plt.plot(actual,label="Actual")

plt.plot(pred_bilstm,label="Bidirectional LSTM")

plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(actual, label="Actual", color="blue", linewidth=2)

plt.plot(pred_lstm,
         label="Predicted",
         color="red",
         linestyle="--",
         linewidth=1)

plt.legend()
plt.title("LSTM Prediction")
plt.grid(True)
plt.show()